# Week 6 — Metrics, Imbalance & Generalization
**Machine Learning for the Natural Sciences**

Until now we've mostly used **accuracy** to evaluate models. But accuracy
can be misleading — especially when classes are imbalanced. A model that
predicts "no landslide" 99% of the time might be 99% accurate but
completely useless for hazard mapping.

This week is about building **trustworthy** models.

**What you'll learn:**
- Precision, recall, F1-score — and when each matters
- Confusion matrix deep dive: false positives vs. false negatives
- Handling imbalanced data: class weights, SMOTE
- Cross-validation strategies: k-fold, stratified, leave-one-out
- Overfitting diagnostics: learning curves

---

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, cross_val_score,
                                      StratifiedKFold, learning_curve)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, ConfusionMatrixDisplay,
                              classification_report, roc_curve, auc)

RANDOM_STATE = 42

## 1. Why Accuracy Lies: A Thought Experiment
Imagine you're predicting landslides. Out of 1,000 sites:
- 990 are stable (no landslide)
- 10 are landslide sites

A model that ALWAYS predicts "stable" gets 99% accuracy.
But it catches 0% of the landslides — it's useless.

In [ ]:
# Let's simulate this
y_true = np.array([0]*990 + [1]*10)
y_dumb = np.zeros(1000, dtype=int)  # always predicts "stable"

print(f"Accuracy of 'always stable': {accuracy_score(y_true, y_dumb):.1%}")
print(f"Precision for landslide class: {precision_score(y_true, y_dumb, zero_division=0):.1%}")
print(f"Recall for landslide class: {recall_score(y_true, y_dumb):.1%}")
print(f"\nThis model catches ZERO landslides. Accuracy is lying to us.")

## 2. Precision vs. Recall
- **Precision** = Of everything the model labeled positive, how many
  actually were? (Are we crying wolf?)
- **Recall** = Of everything that was actually positive, how many did
  the model catch? (Are we missing real events?)
- **F1** = Harmonic mean of precision and recall (balanced summary)

Which matters more depends on your problem:
- Landslide warning? **Recall** — missing a real landslide is catastrophic
- Mineral exploration? **Precision** — drilling is expensive, minimize false leads
- Species ID? Usually **F1** — a balanced measure

## 3. Penguins: Creating an Imbalanced Dataset
Penguins is naturally balanced. Let's deliberately imbalance it to
see how metrics change.

In [ ]:
df = sns.load_dataset("penguins").dropna()
le = LabelEncoder()

# Create imbalance: keep all Adelie and Gentoo, but only 10 Chinstrap
df_adelie = df[df["species"] == "Adelie"]
df_gentoo = df[df["species"] == "Gentoo"]
df_chinstrap = df[df["species"] == "Chinstrap"].sample(10, random_state=RANDOM_STATE)

df_imb = pd.concat([df_adelie, df_gentoo, df_chinstrap])
print("Imbalanced class distribution:")
print(df_imb["species"].value_counts())

In [ ]:
feature_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X_imb = df_imb[feature_cols]
y_imb = le.fit_transform(df_imb["species"])

X_train, X_test, y_train, y_test = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=RANDOM_STATE, stratify=y_imb
)

print(f"\nTraining set distribution: {np.bincount(y_train)}")
print(f"Test set distribution:     {np.bincount(y_test)}")

## 4. Baseline: Ignoring the Imbalance

In [ ]:
rf_base = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_base.fit(X_train, y_train)
y_pred_base = rf_base.predict(X_test)

print("=== Baseline (no imbalance handling) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_base):.3f}")
print(f"\n{classification_report(y_test, y_pred_base, target_names=le.classes_)}")

In [ ]:
ConfusionMatrixDisplay.from_estimator(rf_base, X_test, y_test,
                                       display_labels=le.classes_, cmap="Blues")
plt.title("Baseline — No Imbalance Handling")
plt.tight_layout()
plt.show()

### 🔍 Your Turn
**TODO:** Look at the per-class metrics. What happened to Chinstrap's
recall? Why?

*Your answer:*

## 5. Strategy 1: Class Weights
Tell the model that rare classes are more important by weighting
misclassification of rare samples more heavily.

In [ ]:
rf_weighted = RandomForestClassifier(
    n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE
)
rf_weighted.fit(X_train, y_train)
y_pred_w = rf_weighted.predict(X_test)

print("=== With class_weight='balanced' ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_w):.3f}")
print(f"\n{classification_report(y_test, y_pred_w, target_names=le.classes_)}")

## 6. Strategy 2: SMOTE (Synthetic Minority Oversampling)
SMOTE creates synthetic examples of the minority class by interpolating
between existing minority samples.

**Install:** `pip install imbalanced-learn`

In [ ]:
# !pip install imbalanced-learn

try:
    from imblearn.over_sampling import SMOTE

    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

    print(f"Before SMOTE: {np.bincount(y_train)}")
    print(f"After SMOTE:  {np.bincount(y_train_sm)}")

    rf_smote = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
    rf_smote.fit(X_train_sm, y_train_sm)
    y_pred_sm = rf_smote.predict(X_test)

    print(f"\n=== With SMOTE ===")
    print(f"Accuracy: {accuracy_score(y_test, y_pred_sm):.3f}")
    print(f"\n{classification_report(y_test, y_pred_sm, target_names=le.classes_)}")

except ImportError:
    print("imbalanced-learn not installed. Run: pip install imbalanced-learn")

## 7. Learning Curves: Diagnosing Over/Underfitting

In [ ]:
# Use the balanced dataset for this section
df_full = sns.load_dataset("penguins").dropna()
X_full = df_full[feature_cols]
y_full = le.fit_transform(df_full["species"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, model) in zip(axes, [
    ("Random Forest", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)),
    ("Logistic Reg", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
]):
    train_sizes, train_scores, test_scores = learning_curve(
        model, StandardScaler().fit_transform(X_full) if "Logistic" in name else X_full,
        y_full, cv=5, scoring="accuracy",
        train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
    )

    ax.plot(train_sizes, train_scores.mean(axis=1), "o-", label="Training")
    ax.plot(train_sizes, test_scores.mean(axis=1), "s-", label="Validation")
    ax.fill_between(train_sizes,
                     train_scores.mean(axis=1) - train_scores.std(axis=1),
                     train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.1)
    ax.fill_between(train_sizes,
                     test_scores.mean(axis=1) - test_scores.std(axis=1),
                     test_scores.mean(axis=1) + test_scores.std(axis=1), alpha=0.1)
    ax.set_xlabel("Training Set Size")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"Learning Curve: {name}")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 🔍 Your Turn
**TODO:**
1. A model is **overfitting** when training accuracy is much higher than
   validation accuracy. Which model shows more overfitting?

   *Your answer:*

2. If the learning curves are still converging at the rightmost point,
   what does that suggest? (Would more data help?)

   *Your answer:*

3. For your Data Path dataset, would you expect class imbalance?
   Which strategy (class weights, SMOTE, or both) would you try first?

   *Your answer:*

---
## What to Submit
No separate lab submission this week — the notebook above is a
reference for the techniques. Submit your **Project Proposal** this week.

## What's Next
**Week 7** unifies all **unsupervised learning**: K-Means clustering,
Gaussian Mixture Models, and PCA dimensionality reduction.
**Data Adventure 3** is also due in Week 7 — you'll apply PCA,
clustering, and the metrics you learned this week to your Data Path.